In [ ]:
# ============================================================
# Hybrid LSTM-GARCH — Step 3: GARCH(1,1) Model
# ============================================================

from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

# ── Scale returns to percentage (arch package convention) ─────
ret_pct = returns * 100

In [ ]:
# ── Fit GARCH(1,1) with Student-t distribution ───────────────
# Student-t justified by kurtosis of 11 — fat tails confirmed
print("Fitting GARCH(1,1) with Student-t errors...")
print("(This may take 10-20 seconds)\n")

garch_model = arch_model(
    ret_pct,
    vol='Garch',      # standard GARCH
    p=1,              # lag order for squared returns (ARCH term)
    q=1,              # lag order for variance (GARCH term)
    dist='t'          # Student-t for fat tails
)

garch_result = garch_model.fit(
    update_freq=0,    # suppress iteration output
    disp='off'
)

print(garch_result.summary())


In [ ]:
# ── Extract key parameters ────────────────────────────────────
params = garch_result.params
omega   = params['omega']
alpha   = params['alpha[1]']
beta    = params['beta[1]']
nu      = params['nu']          # degrees of freedom for Student-t

print("\n=== GARCH(1,1) Parameter Interpretation ===")
print(f"omega (ω):   {omega:.6f}  — long-run variance baseline")
print(f"alpha (α):   {alpha:.4f}   — reaction to market shocks (ARCH term)")
print(f"beta  (β):   {beta:.4f}   — volatility persistence (GARCH term)")
print(f"α + β:       {alpha+beta:.4f}   — total persistence (closer to 1 = longer memory)")
print(f"nu:          {nu:.4f}   — Student-t degrees of freedom (fat tails)")
print(f"\nLong-run annualised volatility: {np.sqrt(omega/(1-alpha-beta)*252):.2f}%")
print(f"Half-life of volatility shock:  {np.log(0.5)/np.log(alpha+beta):.1f} days")

In [ ]:
# ── Extract conditional volatility and residuals ──────────────
cond_vol       = garch_result.conditional_volatility  # daily vol in %
std_residuals  = garch_result.std_resid               # standardised residuals

print(f"\n=== Conditional Volatility Summary ===")
print(f"Mean conditional vol:   {cond_vol.mean():.4f}%/day  ({cond_vol.mean()*np.sqrt(252):.2f}% annualised)")
print(f"Max conditional vol:    {cond_vol.max():.4f}%/day  on {cond_vol.idxmax().date()}")
print(f"Min conditional vol:    {cond_vol.min():.4f}%/day  on {cond_vol.idxmin().date()}")

In [ ]:
# ── Save GARCH outputs for LSTM step ─────────────────────────
garch_df = pd.DataFrame({
    'Date':           returns.index,
    'Log_Return':     returns.values,
    'Cond_Vol':       cond_vol.values / 100,     # back to decimal
    'Std_Residual':   std_residuals.values
}).set_index('Date')

garch_df.to_csv('data/garch_outputs.csv')
print("\nGARCH outputs saved to data/garch_outputs.csv")
print(garch_df.head())

In [ ]:
# ── Plot 04: GARCH conditional volatility ─────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Panel 1: Returns vs conditional volatility bands
ax1 = axes[0]
ax1.plot(returns.index, returns * 100, color='#2166ac',
         linewidth=0.4, alpha=0.7, label='Log returns (%)')
ax1.plot(cond_vol.index, cond_vol * 2, color='#d6604d',
         linewidth=1.2, alpha=0.9, label='±2σ GARCH conditional vol')
ax1.plot(cond_vol.index, -cond_vol * 2, color='#d6604d', linewidth=1.2, alpha=0.9)
ax1.fill_between(cond_vol.index, cond_vol * 2, -cond_vol * 2,
                 alpha=0.08, color='#d6604d')
ax1.set_title('GARCH(1,1) Conditional Volatility Bands vs Actual Returns',
              fontweight='bold', fontsize=12)
ax1.set_ylabel('Return / Volatility (%)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Panel 2: Conditional volatility over time
ax2 = axes[1]
ann_vol = cond_vol * np.sqrt(252)
ax2.fill_between(ann_vol.index, ann_vol, alpha=0.35, color='#d6604d')
ax2.plot(ann_vol.index, ann_vol, color='#d6604d', linewidth=0.7)
ax2.axhline(ann_vol.mean(), color='black', linestyle='--',
            linewidth=1, label=f'Mean: {ann_vol.mean():.1f}%')
ax2.set_title('GARCH(1,1) Annualised Conditional Volatility (1993–2024)',
              fontweight='bold', fontsize=12)
ax2.set_ylabel('Annualised Volatility (%)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# Panel 3: Standardised residuals
ax3 = axes[2]
ax3.plot(std_residuals.index, std_residuals,
         color='#4d9221', linewidth=0.4, alpha=0.7)
ax3.axhline(0,  color='black', linewidth=0.5)
ax3.axhline(3,  color='red', linewidth=0.8,
            linestyle='--', label='±3σ threshold')
ax3.axhline(-3, color='red', linewidth=0.8, linestyle='--')
ax3.set_title('GARCH Standardised Residuals\n(should be roughly i.i.d. — extreme values = crisis events)',
              fontweight='bold', fontsize=12)
ax3.set_ylabel('Standardised Residual')
ax3.set_xlabel('Date')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/04_garch_volatility.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/04_garch_volatility.png")

# ── Plot 05: Residual diagnostics ─────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

# ACF of standardised residuals
acf_resid = acf(std_residuals.dropna(), nlags=40, alpha=0.05)
lags_r = range(len(acf_resid[0]))
ci_r_lo = [acf_resid[1][i][0] - acf_resid[0][i] for i in lags_r]
ci_r_hi = [acf_resid[1][i][1] - acf_resid[0][i] for i in lags_r]
axes2[0].bar(lags_r, acf_resid[0], color='#4d9221', alpha=0.7, width=0.6)
axes2[0].fill_between(lags_r, ci_r_lo, ci_r_hi, alpha=0.2, color='gray')
axes2[0].axhline(0, color='black', linewidth=0.5)
axes2[0].set_title('ACF of Standardised Residuals\n(should show no pattern if GARCH fits well)',
                   fontweight='bold', fontsize=11)
axes2[0].set_xlabel('Lag (days)')
axes2[0].set_ylabel('Autocorrelation')
axes2[0].grid(alpha=0.3)

In [ ]:
# ACF of squared standardised residuals
acf_resid_sq = acf(std_residuals.dropna()**2, nlags=40, alpha=0.05)
lags_rs = range(len(acf_resid_sq[0]))
ci_rs_lo = [acf_resid_sq[1][i][0] - acf_resid_sq[0][i] for i in lags_rs]
ci_rs_hi = [acf_resid_sq[1][i][1] - acf_resid_sq[0][i] for i in lags_rs]
axes2[1].bar(lags_rs, acf_resid_sq[0], color='#762a83', alpha=0.7, width=0.6)
axes2[1].fill_between(lags_rs, ci_rs_lo, ci_rs_hi, alpha=0.2, color='gray')
axes2[1].axhline(0, color='black', linewidth=0.5)
axes2[1].set_title('ACF of Squared Standardised Residuals\n(should be flat — ARCH effects removed by GARCH)',
                   fontweight='bold', fontsize=11)
axes2[1].set_xlabel('Lag (days)')
axes2[1].set_ylabel('Autocorrelation')
axes2[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/05_garch_residual_diagnostics.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/05_garch_residual_diagnostics.png")